### LIBRARY IMPORTS

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor
from src.gb_classifier import GBClassifier

warnings.simplefilter(action='ignore', category=FutureWarning)

### CONFIGURATION

In [4]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_processed_data()

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)
X_test, y_test = processor.split_features_target(test)

y_train, y_valid, y_test = processor.transform_target(y_train, y_valid, y_test)

### LOGISTIC REGRESSION

In [14]:
%%time

logistic = LogisticRegression(C=1.0, random_state=42)
logistic.fit(X_train, y_train)

logistic_valid_preds = logistic.predict(X_valid)
logistic_valid_probs = logistic.predict_proba(X_valid)

logistic_test_preds = logistic.predict(X_test)
logistic_test_probs = logistic.predict_proba(X_test)

print(f"Logistic validation log loss: {log_loss(y_valid, logistic_valid_probs):.4f}")
print(f"Logistic validation accuracy: {accuracy_score(y_valid, logistic_valid_preds):.4f}")
print('-' * 50)
print(f"Logistic test log loss: {log_loss(y_test, logistic_test_probs):.4f}")
print(f"Logistic test accuracy: {accuracy_score(y_test, logistic_test_preds):.4f}")
print('-' * 50)

Logistic validation log loss: 0.2713
Logistic validation accuracy: 0.8870
--------------------------------------------------
Logistic test log loss: 0.2747
Logistic test accuracy: 0.8848
--------------------------------------------------
CPU times: total: 688 ms
Wall time: 148 ms


### RANDOM FOREST

In [16]:
%%time

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

rf_valid_preds = rf.predict(X_valid)
rf_valid_probs = rf.predict_proba(X_valid)

rf_test_preds = rf.predict(X_test)
rf_test_probs = rf.predict_proba(X_test)

print(f"RF validation log loss: {log_loss(y_valid, rf_valid_probs):.4f}")
print(f"RF validation accuracy: {accuracy_score(y_valid, rf_valid_preds):.4f}")
print('-' * 50)
print(f"RF test log loss: {log_loss(y_test, rf_test_probs):.4f}")
print(f"RF test accuracy: {accuracy_score(y_test, rf_test_preds):.4f}")
print('-' * 50)

RF validation log loss: 0.2887
RF validation accuracy: 0.8817
--------------------------------------------------
RF test log loss: 0.2921
RF test accuracy: 0.8801
--------------------------------------------------
CPU times: total: 6.61 s
Wall time: 6.6 s


### NEURAL NETWORK

In [20]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        input_size = X_train.shape[1]
        
        unique_classes = np.unique(y_train)
        output_size = len(unique_classes)
        
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.long).to(self.device).squeeze()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            print(f"Epoch: {epoch + 1} | Validation log loss: {val_loss:.4f} | Validation accuracy {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [5]:
mlp = MLP(epochs=100, learning_rate=0.0001, hidden_size=[64, 32], batch_size=256)
mlp.fit(X_train, y_train, X_valid, y_valid)

Epoch: 1 | Validation log loss: 0.3928 | Validation accuracy 0.8753
Epoch: 2 | Validation log loss: 0.2832 | Validation accuracy 0.8860
Epoch: 3 | Validation log loss: 0.2763 | Validation accuracy 0.8881
Epoch: 4 | Validation log loss: 0.2746 | Validation accuracy 0.8886
Epoch: 5 | Validation log loss: 0.2740 | Validation accuracy 0.8885
Epoch: 6 | Validation log loss: 0.2738 | Validation accuracy 0.8888
Epoch: 7 | Validation log loss: 0.2734 | Validation accuracy 0.8890
Epoch: 8 | Validation log loss: 0.2732 | Validation accuracy 0.8894
Epoch: 9 | Validation log loss: 0.2731 | Validation accuracy 0.8893
Epoch: 10 | Validation log loss: 0.2732 | Validation accuracy 0.8885
Epoch: 11 | Validation log loss: 0.2731 | Validation accuracy 0.8885
Epoch: 12 | Validation log loss: 0.2728 | Validation accuracy 0.8898
Epoch: 13 | Validation log loss: 0.2727 | Validation accuracy 0.8892
Epoch: 14 | Validation log loss: 0.2731 | Validation accuracy 0.8887
Epoch: 15 | Validation log loss: 0.2727 | V